In [ ]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML, clear_output
import glob

# Path to your data directory
DATA_DIR = 'data'

# Get all CSV files in the directory
csv_files = glob.glob(os.path.join(DATA_DIR, '*.csv'))
csv_files.sort()  # Sort files alphabetically

output = []

output.append(f"Found {len(csv_files)} CSV files in {DATA_DIR}")

# Function to generate basic info for each CSV file
def explore_csv(file_path):
    file_name = os.path.basename(file_path)
    output.append(f"\n{'='*80}\n{file_name}\n{'='*80}")
    
    try:
        # Read the CSV file
        df = pd.read_csv(file_path)
        
        # Basic file info
        file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
        output.append(f"File size: {file_size_mb:.2f} MB")
        output.append(f"Shape: {df.shape} (rows, columns)")
        
        # Display column names and types
        output.append("\nColumns and data types:")
        output.append(df.dtypes.to_string())
        
        # Check for missing values
        missing = df.isnull().sum()
        if missing.sum() > 0:
            output.append("\nMissing values:")
            output.append(missing[missing > 0].to_string())
        else:
            output.append("\nNo missing values")
        
        # Show the first 5 rows
        output.append("\nSample data (first 5 rows):")
        output.append(df.head().to_html())
        
        # Basic statistics for numeric columns
        if df.select_dtypes(include=np.number).shape[1] > 0:
            output.append("\nBasic statistics for numeric columns:")
            output.append(df.describe().to_html())
        
        # For categorical columns, show unique values count
        categorical_cols = df.select_dtypes(include=['object']).columns
        if len(categorical_cols) > 0:
            output.append("\nCategorical columns unique value counts:")
            for col in categorical_cols:
                unique_count = df[col].nunique()
                output.append(f"{col}: {unique_count} unique values")
                
                # If fewer than 20 unique values, show the distribution
                if unique_count < 20 and unique_count > 0:
                    output.append(df[col].value_counts().head(10).to_string())
                    output.append("")
        
        # Return basic info for summary
        return {
            'file_name': file_name,
            'rows': df.shape[0],
            'columns': df.shape[1],
            'size_mb': file_size_mb,
            'column_names': list(df.columns)
        }
    
    except Exception as e:
        output.append(f"Error exploring {file_name}: {str(e)}")
        return {
            'file_name': file_name,
            'error': str(e)
        }

# Explore each CSV file and collect summary info
summaries = []
for file_path in csv_files:
    summary = explore_csv(file_path)
    summaries.append(summary)

# Create a summary DataFrame
summary_df = pd.DataFrame(summaries)
output.append("\n\n")
output.append("="*80)
output.append("OVERALL SUMMARY OF ALL CSV FILES")
output.append("="*80)
output.append(summary_df[['file_name', 'rows', 'columns', 'size_mb']].to_html())

# Generate a visual overview of the data files
plt.figure(figsize=(12, 8))
summary_df = summary_df.sort_values('rows', ascending=False)
bars = plt.barh(summary_df['file_name'], summary_df['rows'])
plt.xscale('log')
plt.xlabel('Number of Rows (log scale)')
plt.ylabel('File Name')
plt.title('Size Comparison of All CSV Files')
plt.tight_layout()
plt.show()

# Print key files to focus on
output.append("\nKey files for analysis based on size and content:")
for i, row in summary_df.head(5).iterrows():
    output.append(f"- {row['file_name']}: {row['rows']} rows, {row['columns']} columns")

# Display all output in a single cell
clear_output()
display(HTML("<br>".join(output)))

: 

Here's an analysis of the dataset usage and opportunities for improvement:

### Files Used in Current Model & Why:
1. **Core Team Data**
   - `MTeams.csv`/`WTeams.csv` (Team IDs and names)
   - *Why*: Essential for identifying teams and their divisions

2. **Game Results**
   - `MRegularSeasonCompactResults.csv`/`WRegularSeasonCompactResults.csv`
   - `MNCAATourneyCompactResults.csv`/`WNCAATourneyCompactResults.csv`
   - *Why*: Foundation for calculating team statistics and win probabilities

3. **Tournament Structure**
   - `MNCAATourneySeeds.csv`/`WNCAATourneySeeds.csv`
   - *Why*: Critical seed information for matchup predictions

4. **Rankings**
   - `MMasseyOrdinals.csv`
   - *Why*: Provides valuable team ranking data over time

5. **Submission Templates**
   - `SampleSubmissionStage1.csv`
   - *Why*: Required format for final predictions

### Key Unused Files & Potential Benefits:

1. **Conference Data** (`Conferences.csv`, `MTeamConferences.csv`, `WTeamConferences.csv`)
   - *Potential Benefit*: Model conference strength and intra-conference rivalry effects
   - *Challenge*: Requires conference performance metrics over time

2. **Geographic Data** (`Cities.csv`, `MGameCities.csv`, `WGameCities.csv`)
   - *Potential Benefit*: Add travel distance calculations and home court advantage
   - *Challenge*: Requires geocoding and distance matrix calculations

3. **Detailed Box Scores** (`MRegularSeasonDetailedResults.csv`, `WNCAATourneyDetailedResults.csv`, etc.)
   - *Potential Benefit*: Enable advanced metrics like:
   ```python
   # Example: Calculate effective field goal percentage
   df['EFG%'] = (df['WFGM'] + 0.5 * df['WFGM3']) / df['WFGA']
   ```

4. **Conference Tournament Results** (`MConferenceTourneyGames.csv`, `WConferenceTourneyGames.csv`)
   - *Potential Benefit*: Capture late-season momentum and tournament pressure experience

5. **Coach Data** (`MTeamCoaches.csv`)
   - *Potential Benefit*: Add coaching experience metrics:
   ```python
   # Example: Coach tenure calculation
   coach_tenure = df.groupby('CoachName')['Season'].nunique()
   ```

6. **Secondary Tournaments** (`MSecondaryTourney*.csv`, `WSecondaryTourney*.csv`)
   - *Potential Benefit*: More data points for teams not making main tournament

### Strategic Unused Files:
1. **Season Regions** (`MSeasons.csv`, `WSeasons.csv`)
   - *Potential Use*: Account for regional biases in tournament selection

2. **Team Spellings** (`MTeamSpellings.csv`, `WTeamSpellings.csv`)
   - *Current Issue*: Encoding errors need fixing first
   - *Potential Use*: Connect with external data sources

3. **Game Slots** (`MNCAATourneySlots.csv`, `WNCAATourneySlots.csv`)
   - *Potential Use*: Model bracket progression probabilities

### Implementation Considerations:

1. **Feature Engineering Pipeline**:
```python
def create_advanced_features(df):
    # Conference strength
    df['ConfStrength'] = df.groupby('ConfAbbrev')['WinPct'].transform('mean')
    
    # Travel distance calculation
    df['TravelDistance'] = calculate_distance(df['TeamCity'], df['GameCity'])
    
    # Coaching experience
    df['CoachExperience'] = df.groupby('CoachName')['Season'].cumcount()
    
    return df
```

2. **Data Enrichment Strategy**:
   - Phase 1: Add conference metrics
   - Phase 2: Incorporate geographic features
   - Phase 3: Integrate detailed box score stats
   - Phase 4: Add coaching/schedule features

3. **Computational Tradeoffs**:
   - Using detailed box scores would increase feature count from ~15 to 50+
   - Coach data would add ~5 new temporal features
   - Geographic features require external API calls for geocoding

### Recommended Next Steps:
1. **Priority Additions**:
   ```python
   # Start with conference data integration
   conferences = pd.read_csv('data/MTeamConferences.csv')
   merged_data = game_data.merge(conferences, on=['Season', 'TeamID'])
   ```

2. **Gradual Implementation**:
   - Add 2-3 feature categories at a time
   - Monitor validation score improvements
   - Maintain separate gender models during experimentation

3. **Error Handling**:
   ```python
   # Fix team spellings encoding first
   try:
       spellings = pd.read_csv('data/MTeamSpellings.csv', encoding='latin1')
   except UnicodeDecodeError:
       # Implement fallback encoding strategy
   ```

This phased approach allows for controlled complexity while maximizing the value of underutilized data sources. Would you like me to elaborate on implementing any specific feature category?